## exploration_renvois_orphelins_bofip

**Fichier(s) source :** `./data/inventaire_bofip_stock_live_20260521.csv` (inventaire produit par `profilage_bofip_consolide.ipynb`) et `./data/bofip_stock_live_20260521.tgz` (stock du 21.05.2026)

**Fichier(s) de sortie :** DataFrame en mémoire

**Description :** Exploration des renvois (dc:relation) des documents Contenu du stock BOFiP : volume total, documents sans renvoi, orphelins par type (Actualité, Contenu, Fichier, Autre), et identification des orphelins « Autre » par relecture des renvois bruts dans l'archive.

In [1]:
import pandas as pd

CSV = r"./data/inventaire_bofip_stock_live_20260521.csv"

df = pd.read_csv(CSV)
print(f"Documents chargés : {len(df)}")

Documents chargés : 6311


In [2]:
# === Comptages généraux ===
tot_r = int(df["nb_renvois"].sum())
tot_o = int(df["nb_orphelins"].sum())
sans_renvoi = int((df["nb_renvois"] == 0).sum())
avec_orphelin = int((df["nb_orphelins"] > 0).sum())

print(f"Total renvois : {tot_r}")
print(f"Documents sans aucun renvoi : {sans_renvoi} ({sans_renvoi/len(df):.1%})")
print(f"Documents avec au moins un orphelin : {avec_orphelin}")
print(f"Total orphelins : {tot_o} ({tot_o/tot_r:.1%})")

Total renvois : 23128
Documents sans aucun renvoi : 1250 (19.8%)
Documents avec au moins un orphelin : 1283
Total orphelins : 1354 (5.9%)


In [3]:
# === Orphelins par type ===
orph_actu = int(df["orph_actualite"].sum())
orph_cont = int(df["orph_contenu"].sum())
orph_fich = int(df["orph_fichier"].sum())
orph_autre = tot_o - orph_actu - orph_cont - orph_fich

print("Orphelins par type :")
print(f"  Actualité : {orph_actu}")
print(f"  Contenu   : {orph_cont}")
print(f"  Fichier   : {orph_fich}")
print(f"  Autre     : {orph_autre}")
print(f"  Total     : {orph_actu + orph_cont + orph_fich + orph_autre}")

if orph_autre > 0:
    print(f"\nAttention : {orph_autre} orphelins ne sont ni Actualité, ni Contenu, ni Fichier.")
    print("Identification ci-dessous.")

Orphelins par type :
  Actualité : 1071
  Contenu   : 265
  Fichier   : 15
  Autre     : 3
  Total     : 1354

Attention : 3 orphelins ne sont ni Actualité, ni Contenu, ni Fichier.
Identification ci-dessous.


In [4]:
# === Identification des orphelins 'Autre' ===
# Pour les trouver, il faut relire les renvois bruts depuis l'archive
# On identifie les documents dont nb_orphelins > orph_actualite + orph_contenu + orph_fichier

df["orph_autre"] = df["nb_orphelins"] - df["orph_actualite"] - df["orph_contenu"] - df["orph_fichier"]
docs_autre = df[df["orph_autre"] > 0]

print(f"Documents avec des orphelins 'Autre' : {len(docs_autre)}")
if len(docs_autre) > 0:
    print()
    print(docs_autre[["identifiant", "serie", "type", "nb_renvois", "nb_orphelins", 
                       "orph_actualite", "orph_contenu", "orph_fichier", "orph_autre"]].to_string(index=False))

Documents avec des orphelins 'Autre' : 1

identifiant serie        type  nb_renvois  nb_orphelins  orph_actualite  orph_contenu  orph_fichier  orph_autre
   2064-PGP    IS Commentaire           6             3               0             0             0           3


In [5]:
# === Relecture des renvois bruts pour identifier les orphelins 'Autre' ===
import tarfile, os, re

ARCHIVE = r"./data/bofip_stock_live_20260521.tgz"

if len(docs_autre) > 0:
    ids_autre = set(docs_autre["identifiant"].values)
    print(f"Recherche des renvois bruts pour {len(ids_autre)} documents...")
    
    # Construire l'ensemble des identifiants présents
    presents = set()
    renvois_bruts = []
    
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        par_dossier = {}
        for m in tar.getmembers():
            if not m.isfile():
                continue
            base = os.path.basename(m.name)
            dossier = os.path.dirname(m.name)
            par_dossier.setdefault(dossier, {})[base] = m
            if base == "document.xml":
                segs = dossier.split("/")
                if len(segs) >= 2:
                    presents.add(segs[-2])
        
        # Relire les document.xml des documents concernés
        for dossier, fichiers in par_dossier.items():
            if "document.xml" not in fichiers:
                continue
            segs = dossier.split("/")
            if len(segs) < 2:
                continue
            identifiant = segs[-2]
            if identifiant not in ids_autre:
                continue
            
            import xml.etree.ElementTree as ET
            data = tar.extractfile(fichiers["document.xml"]).read()
            root = ET.fromstring(data)
            for el in root.iter():
                tag = el.tag.rsplit("}", 1)[-1] if "}" in el.tag else el.tag
                if tag == "relation" and el.text:
                    texte = el.text.strip()
                    if ":" in texte:
                        typ, cible = texte.split(":", 1)
                    else:
                        typ, cible = "Inconnu", texte
                    est_present = cible in presents
                    if not est_present and typ not in ("Actualite", "Contenu", "Fichier"):
                        renvois_bruts.append({
                            "source": identifiant,
                            "renvoi_brut": texte,
                            "type_detecte": typ,
                            "cible": cible,
                            "present": est_present
                        })
    
    if renvois_bruts:
        df_bruts = pd.DataFrame(renvois_bruts)
        print(f"\nOrphelins 'Autre' trouvés : {len(df_bruts)}")
        print(df_bruts.to_string(index=False))
    else:
        print("Aucun orphelin 'Autre' trouvé dans les renvois bruts.")
else:
    print("Pas d'orphelins 'Autre' à investiguer.")

Recherche des renvois bruts pour 1 documents...

Orphelins 'Autre' trouvés : 3
  source    renvoi_brut type_detecte    cible  present
2064-PGP Image:7205-PGP        Image 7205-PGP    False
2064-PGP Image:7206-PGP        Image 7206-PGP    False
2064-PGP Image:7207-PGP        Image 7207-PGP    False
